In [48]:
import numpy as np

In [49]:
class LogisticRegression:

    def __init__(
        self, 
        max_iter : int = 1000, 
        alpha : float = 0.05,
        C=1.0, 
        fit_intercept : bool = True
    ):
        """
        alpha : learning rate
        C : inverse of regularization strength (sklearn jaisa). 
            Bada C = kam regularization. C=None -> no regularization.
        fit_intercept : bias term add karna hai ya nahi
        """
        self.alpha = alpha
        self.max_iter = max_iter
        self.C = C
        self.fit_intercept = fit_intercept

    def _add_bias(self, X:np.ndarray) -> np.ndarray:
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            return np.hstack([ones, X])
        return X

    def fit(self, X: np.ndarray, y: np.ndarray):

        X = self._add_bias(X)
        y = y.reshape(-1, 1)

        self.X = X
        self.y = y

        W = np.zeros((X.shape[1], 1))

        for _ in range(self.max_iter):
            W = W - self.alpha * self.gradient(X, y, W)

        self.W = W

    def y_pred(self, X: np.ndarray, W: np.ndarray) -> np.ndarray:
        z = X @ W
        z = np.clip(z, -500, 500)      # overflow guard for exp()
        return 1 / (1 + np.exp(-z))

    def gradient(self, X: np.ndarray, y: np.ndarray, W: np.ndarray) -> np.ndarray:
        m = X.shape[0]
        y_hat = self.y_pred(X, W)
        grad = (X.T @ (y_hat - y)) / m

        if self.C is not None:
            # L2 regularization, bias (W[0]) ko regularize NAHI karte -- sklearn convention
            reg = W.copy()
            if self.fit_intercept:
                reg[0] = 0
            grad = grad + reg / (self.C * m)

        return grad

    def predict_proba(self, X_test: np.ndarray) -> np.ndarray:
        X_test = self._add_bias(X_test)
        return self.y_pred(X_test, self.W)

    def predict(self, X_test: np.ndarray, threshold=0.5) -> np.ndarray:
        probability = self.predict_proba(X_test)
        return (probability > threshold).astype(int).ravel()   # <-- (n,) shape, sklearn jaisa

In [50]:
X_train = np.array([
    [2, 1], [3, 2], [4, 2], [5, 3], [3, 4],
    [6, 2], [7, 3], [4, 5], [5, 5], [6, 4],
    [8, 3], [7, 5], [9, 4], [8, 6], [10, 5],
    [11, 6], [9, 7], [12, 7], [10, 8], [13, 6],
    [14, 8], [12, 9], [15, 7], [13, 10], [16, 9]
])

y_train = np.array([
    [0], [0], [0], [0], [0],
    [0], [0], [0], [0], [0],
    [0], [0], [0], [0], [0],
    [1], [1], [1], [1], [1],
    [1], [1], [1], [1], [1]
])

X_test = np.array([
    [3, 3],
    [5, 4],
    [7, 4],
    [9, 5],
    [11, 7],
    [12, 8],
    [14, 9],
    [15, 10],
    [17, 11],
    [18, 8]
])

y_test = np.array([
    [0],
    [0],
    [0],
    [0],
    [1],
    [1],
    [1],
    [1],
    [1],
    [1]
])

In [ ]:
lr = LogisticRegression()
lr.fit(X_train , y_train)
y_pred = lr.predict(X_test)

In [52]:
y_pred

array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])